# Lab 07 — Iris Deep-Dive — Stats + First Plot
**Statistics for Analysts Track** · Beginner · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute per-species means and standard deviations
2. Measure class separation with Cohen's d
3. Run one-way ANOVA and a t-based 95% confidence interval
4. Publish a pair-plot (seaborn or matplotlib fallback)

## Datasets (this folder)
- `iris.csv` — auto-download from `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv`

## How to run on Google Colab
1. Click **Start Lab** — the hosted notebook opens directly in Colab under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-07-iris-deep-dive"
# Hosted manifest (Admin: replace ORG/REPO once per deployment).
MANIFEST_URL = f"https://raw.githubusercontent.com/ORG/REPO/main/{LAB_ID}/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("iris.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Statistics for Analysts Track: Group Stats, ANOVA, Confidence Intervals

> **Scenario:** Classic `iris.csv` (150 rows, 4 numeric features + species). Compute per-species statistics, a one-way ANOVA on petal length, a 95% CI for virginica sepal width, and publish a pair-plot.
>
> **You will learn:** `groupby.agg`, variance, F-stat intuition, Cohen’s d, confidence intervals, seaborn/matplotlib pair plots.
> **Time:** ~40 minutes. **Level:** Beginner. **Needs:** pandas + scipy + matplotlib/seaborn. **Env:** 🟢 Colab only.

### Stats mental map

| Excel / idea | Python | Output |
|---|---|---|
| AVERAGEIFS by species | `groupby.mean()` | per-species means |
| Spread | `.std(ddof=1)` | sample SD |
| “Do groups differ?” | one-way ANOVA | F, p |
| “How sure of a mean?” | t-based 95% CI | mean ± t·s/√n |
| “How separated?” | Cohen’s d | effect size |

---

### 1. Load data (local first, Colab fallback)

In [ ]:
import math, os
import pandas as pd
from scipy import stats

def load_iris():
    local = "iris.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv",
            local,
        )
    return pd.read_csv(local)

df = load_iris()
print(df.shape)    # (150, 5)
print(df["species"].value_counts())  # 50 each
print(df.head(3))


---

### 2. Per-species means and SDs

In [ ]:
g = df.groupby("species")
means = g[["sepal_length", "sepal_width", "petal_length", "petal_width"]].mean().round(3)
sds   = g[["sepal_length", "sepal_width", "petal_length", "petal_width"]].std().round(3)
print(means)
print(sds)


Expected mean `petal_length`: **setosa 1.462 · versicolor 4.260 · virginica 5.552**.

---

### 3. Which feature best separates setosa vs versicolor?

Cohen’s d = (mean₂ − mean₁) / pooled SD:

In [ ]:
sv = df[df["species"].isin(["setosa", "versicolor"])]
print(f"{'feature':14s} {'setosa':>8s} {'versicolor':>10s} {'gap':>8s} {'cohen_d':>8s}")
for c in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    a = sv.loc[sv["species"] == "setosa", c]
    b = sv.loc[sv["species"] == "versicolor", c]
    pooled = math.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    d = (b.mean() - a.mean()) / pooled
    print(f"{c:14s} {a.mean():8.3f} {b.mean():10.3f} {abs(b.mean()-a.mean()):8.3f} {d:8.3f}")


Expected: **petal_length wins** (gap 2.798, |d| ≈ 7.9) — far larger than sepal measures.

> Rule of thumb: |d| > 0.8 is a “large” separation in behavioural science; 7.9 is enormous (the classes barely overlap).

---

### 4. One-way ANOVA on petal_length

In [ ]:
groups = [df.loc[df["species"] == sp, "petal_length"].values
          for sp in df["species"].unique()]
F, p = stats.f_oneway(*groups)
print(f"F = {F:.3f}, p = {p:.3e}")
# F = 1180.161, p = 2.86e-91


H₀: all three species share the same mean petal length. p ≈ 0 → reject H₀ decisively.

F intuition: `variance_between_groups / variance_within_groups`. Huge F ⇒ between-group gaps dwarf within-group noise.

---

### 5. 95% CI for virginica sepal_width

In [ ]:
x = df.loc[df["species"] == "virginica", "sepal_width"]
n, mean, sd = len(x), x.mean(), x.std(ddof=1)
se = sd / math.sqrt(n)
t = stats.t.ppf(0.975, df=n - 1)
ci = (mean - t * se, mean + t * se)
print(f"n={n} mean={mean:.4f} sd={sd:.4f} t={t:.4f}")
print(f"95% CI = ({ci[0]:.4f}, {ci[1]:.4f})")
# n=50 mean=2.9740 sd=0.3225 t=2.0096
# 95% CI = (2.8823, 3.0657)


---

### 6. Pair plot (seaborn or matplotlib fallback)

In [ ]:
try:
    import seaborn as sns
    import matplotlib.pyplot as plt
    sns.pairplot(df, hue="species", corner=True)
    plt.savefig("iris_pairplot.png", dpi=120)
    print("saved iris_pairplot.png (seaborn)")
except ImportError:
    import matplotlib.pyplot as plt
    feats = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    colors = {"setosa": "tab:red", "versicolor": "tab:blue", "virginica": "tab:green"}
    fig, axes = plt.subplots(4, 4, figsize=(9, 9))
    for i, ri in enumerate(feats):
        for j, ci_ in enumerate(feats):
            ax = axes[i, j]
            if i == j:
                for sp, col in colors.items():
                    ax.hist(df.loc[df.species == sp, ri], alpha=0.5, label=sp, color=col)
            else:
                for sp, col in colors.items():
                    sub = df[df.species == sp]
                    ax.scatter(sub[ci_], sub[ri], s=8, alpha=0.7, color=col)
            if i == 3: ax.set_xlabel(ci_, fontsize=8)
            if j == 0: ax.set_ylabel(ri, fontsize=8)
    fig.tight_layout(); fig.savefig("iris_pairplot.png", dpi=120)
    print("saved iris_pairplot.png (matplotlib)")


Look for: petal_length vs petal_width cleanly splits setosa from the other two.

---

## Exercises (do these!)

### Exercise 1 — Mean petal_length by species
Print mean `petal_length` for each species (3 d.p.).
*Expected: setosa 1.462 · versicolor 4.260 · virginica 5.552.*

<details>
<summary>Hint</summary>

`df.groupby("species")["petal_length"].mean().round(3)`
</details>

### Exercise 2 — Best separator setosa vs versicolor
For each of the 4 features, compute Cohen’s d between setosa and versicolor. Which feature has the largest |d|?
*Expected: petal_length (|d| ≈ 7.90); petal_width second (≈ 6.82).*

<details>
<summary>Hint</summary>

Pooled SD formula in Section 3; compare absolute values.
</details>

### Exercise 3 — 95% CI for virginica sepal_width
Using t with df = n−1, report mean and CI to 4 d.p.
*Expected: mean 2.9740 · 95% CI (2.8823, 3.0657).*

<details>
<summary>Hint</summary>

`mean ± t.ppf(0.975, n-1) * std(ddof=1) / sqrt(n)`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
print(df.groupby("species")["petal_length"].mean().round(3))
# setosa      1.462
# versicolor  4.260
# virginica   5.552

# --- Solution 2 ---
sv = df[df.species.isin(["setosa", "versicolor"])]
best, best_d = None, 0
for c in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    a = sv.loc[sv.species == "setosa", c]
    b = sv.loc[sv.species == "versicolor", c]
    pooled = math.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    d = abs((b.mean()-a.mean())/pooled)
    if d > best_d:
        best, best_d = c, d
print(best, round(best_d, 3))  # petal_length 7.899

# --- Solution 3 ---
x = df.loc[df.species == "virginica", "sepal_width"]
n = len(x); m = x.mean(); s = x.std(ddof=1)
t = stats.t.ppf(0.975, n-1)
lo, hi = m - t*s/math.sqrt(n), m + t*s/math.sqrt(n)
print(f"{m:.4f} ({lo:.4f}, {hi:.4f})")  # 2.9740 (2.8823, 3.0657)


### What to learn next
- Kruskal–Wallis (non-parametric alternative to ANOVA).
- Pairwise t-tests with Bonferroni correction.
- Effect sizes beyond d (η² from ANOVA).
- Then: logistic regression predicting species from measurements (Course 2).
- Cheat sheet: groupby → mean/sd → ANOVA for ≥3 groups → t-CI for one mean → plot to verify.

*Files in this folder: `iris.csv` · output `iris_pairplot.png`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, 
or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
